# Tạo data.yaml cho Ultralytics, chỉ sau PASS
**Mục tiêu:** chỉ tạo `data/data.yaml` khi notebook 10 báo **PASS** cho đúng phiên bản `split_manifest.json` và danh sách ảnh. **Input:** `data/splits/{train,val,test}.txt`, `split_manifest.json`, `docs/leakage_report.md`, Class.txt của PKU và ảnh/label processed. **Output:** `data/data.yaml` (Ultralytics). Nếu FAIL hoặc thiếu báo cáo thì dừng, không tạo YAML. Không train model và không gộp DeepPCB. Chạy sau 08 → 09 → 10.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown
import hashlib, json, re
import pandas as pd
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/raw/PKU-Market-PCB(Data enhanced version)/Class.txt').is_file()), None)
if ROOT is None: raise FileNotFoundError('Chạy trong project PKU')
manifest_path = ROOT / 'data/splits/split_manifest.json'
report_path = ROOT / 'docs/leakage_report.md'
if not manifest_path.is_file() or not report_path.is_file(): raise RuntimeError('FAIL: thiếu manifest hoặc leakage_report.md; phải chạy 08 → 09 → 10 thành công trước')
manifest_bytes = manifest_path.read_bytes()
manifest = json.loads(manifest_bytes)
report = report_path.read_text(encoding='utf-8')
digest = hashlib.sha256(manifest_bytes).hexdigest()
if '**Overall: PASS**' not in report or f'Manifest SHA256: `{digest}`' not in report: raise RuntimeError('FAIL: báo cáo leakage chưa PASS cho manifest hiện tại. Dừng trước khi tạo data.yaml.')
if manifest['dataset'] != 'PKU-Market-PCB(Data enhanced version)': raise ValueError('Manifest phải chỉ chứa PKU')
classes = re.findall(r"['\"]([^'\"]+)['\"]", (ROOT / 'data/raw/PKU-Market-PCB(Data enhanced version)/Class.txt').read_text(encoding='utf-8-sig'))
expected_classes = ['Missing_hole','Mouse_bite','Open_circuit','Short','Spur','Spurious_copper']
if classes != expected_classes: raise ValueError('Class.txt PKU không đúng 6 class ID mong đợi')
display(pd.DataFrame({'class_id': range(len(classes)), 'name_from_Class.txt': classes}))
display(Markdown('Thứ tự class lấy từ Class.txt và đối chiếu 0–5; không remap nhãn DeepPCB.'))


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


,class_id,name_from_Class.txt
0,0,Missing_hole
1,1,Mouse_bite
2,2,Open_circuit
3,3,Short
4,4,Spur
5,5,Spurious_copper


Thứ tự class lấy từ Class.txt và đối chiếu 0–5; không remap nhãn DeepPCB.

In [2]:
split_rows = []
items = manifest['items']
if len({r['image'] for r in items}) != len(items): raise RuntimeError('FAIL: manifest có ảnh trùng')
for split in ('train', 'val', 'test'):
    path = ROOT / 'data/splits' / f'{split}.txt'
    if not path.is_file(): raise FileNotFoundError(path)
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != manifest['lists_sha256'][split]: raise RuntimeError(f'FAIL: {split}.txt thay đổi sau kiểm chứng')
    listed = content.decode('utf-8').splitlines()
    expected_paths = [str((ROOT / row['image']).absolute()) for row in items if row['split'] == split]
    if not listed or listed != expected_paths: raise RuntimeError(f'FAIL: {split}.txt không khớp manifest hoặc rỗng')
    for row in (r for r in items if r['split'] == split):
        image, label, source = (ROOT / row[k] for k in ('image', 'label', 'source_image'))
        if 'DeepPCB' in str(image) or 'DeepPCB' in str(source): raise RuntimeError('FAIL: có DeepPCB trong split chính')
        if not row['image'].startswith('data/processed/images/') or not row['label'].startswith('data/processed/labels/') or not row['source_image'].startswith('data/raw/PKU-Market-PCB(Data enhanced version)/'):
            raise RuntimeError(f'FAIL: path không thuộc PKU processed: {row}')
        if not image.is_symlink() or not source.is_file() or image.resolve() != source.resolve() or not label.is_file() or label.stem != image.stem or label.parent.parent.name != 'labels':
            raise RuntimeError(f'FAIL: đường dẫn ảnh/nhãn sai cho Ultralytics: {image}')
    split_rows.append({'split': split, 'images': len(listed), 'file': str(path.relative_to(ROOT)), 'sha256_matches_reported_manifest': True})
display(pd.DataFrame(split_rows))
display(Markdown('Bảng xác nhận số ảnh, file list và digest đúng manifest đã PASS; ảnh phải nằm trong processed/images song song processed/labels để Ultralytics nhận diện nhãn.'))


,split,images,file,sha256_matches_reported_manifest
0,train,2416,data/splits/train.txt,True
1,val,608,data/splits/val.txt,True
2,test,480,data/splits/test.txt,True


Bảng xác nhận số ảnh, file list và digest đúng manifest đã PASS; ảnh phải nằm trong processed/images song song processed/labels để Ultralytics nhận diện nhãn.

In [3]:
# JSON string là scalar YAML hợp lệ; không cần phụ thuộc PyYAML để tạo file.
yaml_text = '\n'.join([
    f'path: {json.dumps(str(ROOT.absolute()), ensure_ascii=False)}',
    'train: data/splits/train.txt',
    'val: data/splits/val.txt',
    'test: data/splits/test.txt',
    f'nc: {len(classes)}',
    'names:',
    *(f'  {i}: {name}' for i, name in enumerate(classes)),
    ''
])
target = ROOT / 'data/data.yaml'
if target.exists() and target.read_text(encoding='utf-8') != yaml_text: raise FileExistsError(f'Không ghi đè data.yaml có nội dung khác: {target}')
if not target.exists(): target.write_text(yaml_text, encoding='utf-8')
display(Markdown('## Kết quả: `data/data.yaml`'))
display(Markdown('```yaml\n' + yaml_text + '```'))
display(Markdown('**Đạt phase 11:** YAML chỉ trỏ tới split PKU đã PASS leakage và 6 lớp lấy từ Class.txt; không train model.'))


## Kết quả: `data/data.yaml`

```yaml
path: "/Users/duytan/orca/projects/PCB-DetectionImprove"
train: data/splits/train.txt
val: data/splits/val.txt
test: data/splits/test.txt
nc: 6
names:
  0: Missing_hole
  1: Mouse_bite
  2: Open_circuit
  3: Short
  4: Spur
  5: Spurious_copper
```

**Đạt phase 11:** YAML chỉ trỏ tới split PKU đã PASS leakage và 6 lớp lấy từ Class.txt; không train model.

## Nhận xét / điều kiện kết thúc
PASS yêu cầu báo cáo leakage có digest trùng manifest, mọi TXT chưa đổi, ảnh symlink tới PKU và label processed tồn tại cùng stem. Thiếu bất kỳ điều kiện nào: FAIL và không tạo `data/data.yaml`. Nếu phase 10 báo leakage, phải điều tra và sửa split, chạy lại 09–10 rồi mới quay lại 11; không được bỏ qua cổng này.